PCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
SVM: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
Cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate, StratifiedKFold
import warnings
import sklearn.exceptions
warnings.filterwarnings("ignore", category=sklearn.exceptions.UndefinedMetricWarning)

**Data Exploration**

Here we become familiarized with the data and the different properties of the wines that are being measured. 

In [ ]:
red_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv', sep=";")
red_df["type"] = 'red'
red_df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,red
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,red
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,red
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,red


Describing the dataframe gives a more in-depth analysis of the distribution of the data points for each property.

In [ ]:
red_df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [ ]:
white_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv', sep=";")
white_df["type"] = 'white'
white_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,white
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,white
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,white
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white


In [ ]:
white_df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000
mean,6.854788,0.278241,0.334192,6.391415,0.045772,35.308085,138.360657,0.994027,3.188267,0.489847,10.514267,5.877909
std,0.843868,0.100795,0.121020,5.072058,0.021848,17.007137,42.498065,0.002991,0.151001,0.114126,1.230621,0.885639
min,3.800000,0.080000,0.000000,0.600000,0.009000,2.000000,9.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.300000,0.210000,0.270000,1.700000,0.036000,23.000000,108.000000,0.991723,3.090000,0.410000,9.500000,5.000000
50%,6.800000,0.260000,0.320000,5.200000,0.043000,34.000000,134.000000,0.993740,3.180000,0.470000,10.400000,6.000000
75%,7.300000,0.320000,0.390000,9.900000,0.050000,46.000000,167.000000,0.996100,3.280000,0.550000,11.400000,6.000000
max,14.200000,1.100000,1.660000,65.800000,0.346000,289.000000,440.000000,1.038980,3.820000,1.080000,14.200000,9.000000


In [ ]:
# combining the two datasets
df = pd.concat([red_df,white_df])
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [ ]:
# checking tail to confirm white wine data is concatenated
df.tail()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white
4897,6.0,0.21,0.38,0.8,0.020,22.0,98.0,0.98941,3.26,0.32,11.8,6,white


**Preprocessing Dataframe**

We begin the preprocessing of the dataframe. First shuffling the data then removing the "type" and "quality" column. The "type" column is removed because that is the property we want our model to estimate. The "quality" column was also removed because it was deemed subjective and to have poor correlation to a specific type of wine thus just adding unnecessary data to the input potentially interfering with the ML models ability to find more meaningful patterns. Subsequently we scale the data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# shuffling the data
df = df.sample(frac=1)
# dropping the necessary columns from dataframe
X = df.drop(columns = ['type','quality'])
y_type = df['type']
label_encoder = LabelEncoder()
encoded_y = label_encoder.fit_transform(y_type)

# training-testing data splitting: 80% training 20% testing
type_X_train, type_X_test, type_y_train, type_y_test = train_test_split(X,encoded_y,test_size=0.2)

**Feature Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

# rescale the data to have mean ~ 0 and std-dev ~ 1
scaler=StandardScaler()
scaler_X=scaler.fit_transform(X)
scaler_X

array([[-0.16608919,  0.06277343, -0.05941375, ..., -0.23947061,
        -0.41176462, -0.91546416],
       [ 0.52817634, -0.30169391,  1.17934553, ...,  0.56911368,
         0.9323718 , -0.32852111],
       [ 1.14530125,  1.21692001, -0.40351355, ...,  0.32031851,
         0.46192405, -0.99931317],
       ...,
       [-1.09177657, -0.66616126, -0.81643332, ..., -0.05287424,
        -0.94941918, -0.2446721 ],
       [ 0.8367388 , -0.54467214,  0.07822617, ..., -2.04323557,
        -0.34455779, -0.83161516],
       [-1.16891718, -0.72690581, -0.6787934 , ...,  0.13372214,
        -0.34455779, -1.2508602 ]])

**Precision**: How many predicted positives are actually positive?

**Recall**: How many actual positives were correctly predicted?

**F1 Score**: Harmonic mean of precision and recall (balance between the two).

**ROC AUC Score**: Area Under the Receiver Operating Characteristic Curve, measures the ability of the model to distinguish between classes using probabilities.

**Random Forest Model - Type Classification**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(type_X_train,type_y_train)

y_pred = rf.predict(type_X_test)
y_pred_proba = rf.predict_proba(type_X_test)[:, 1]

precision = precision_score(type_y_test,y_pred)
recall = recall_score(type_y_test,y_pred)
f1 = f1_score(type_y_test,y_pred)
roc_auc = roc_auc_score(type_y_test,y_pred_proba)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)

Precision: 0.997020854021847
Recall: 0.9990049751243781
F1 score: 0.9980119284294235
ROC AUC score: 0.9973016274559405


In [ ]:
rf = RandomForestClassifier()

score = cross_validate(rf,X,encoded_y,scoring=['precision','recall','f1','roc_auc'],cv=10)
print("precision: ",score['test_precision'].mean()*100)
print("recall: ",score['test_recall'].mean()*100)
print("f1: ",score['test_f1'].mean()*100)
print("ROC: ",score['test_roc_auc'].mean()*100)

precision:  99.4726006510932
recall:  99.85705938817246
f1:  99.66406239858546
ROC:  99.76946079783345


**SVM - Type Classification**

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='linear',probability=True)

svm.fit(type_X_train,type_y_train)

y_pred = svm.predict(type_X_test)
y_pred_proba = svm.predict_proba(type_X_test)[:, 1]

precision = precision_score(type_y_test,y_pred)
recall = recall_score(type_y_test,y_pred)
f1 = f1_score(type_y_test,y_pred)
roc_auc = roc_auc_score(type_y_test,y_pred_proba)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)

Precision: 0.984
Recall: 0.992936427850656
F1 score: 0.9884480160723255
ROC AUC score: 0.9925510827218429


In [ ]:
svm = SVC(kernel='linear',probability=True)

score = cross_validate(svm,X,encoded_y,scoring=['precision','recall','f1','roc_auc'],cv=10)
print("precision: ",score['test_precision'].mean()*100)
print("recall: ",score['test_recall'].mean()*100)
print("f1: ",score['test_f1'].mean()*100)
print("ROC: ",score['test_roc_auc'].mean()*100)

precision:  98.926439621182
recall:  99.428279287175
f1:  99.17554468608554
ROC:  99.47302582596876


**K-Nearest Neighbors - Type Classification**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(weights='distance',metric='euclidean')

knn.fit(type_X_train,type_y_train)

y_pred = knn.predict(type_X_test)
y_pred_proba = knn.predict_proba(type_X_test)[:, 1]

precision = precision_score(type_y_test,y_pred)
recall = recall_score(type_y_test,y_pred)
f1 = f1_score(type_y_test,y_pred)
roc_auc = roc_auc_score(type_y_test,y_pred_proba)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)

Precision: 0.9761668321747765
Recall: 0.9781094527363184
F1 score: 0.9771371769383699
ROC AUC score: 0.9719082553335019


In [ ]:
knn = KNeighborsClassifier(weights='distance',metric='euclidean')

score = cross_validate(knn,scaler_X,encoded_y,scoring=['precision','recall','f1','roc_auc'],cv=10)
print("precision: ",score['test_precision'].mean()*100)
print("recall: ",score['test_recall'].mean()*100)
print("f1: ",score['test_f1'].mean()*100)
print("ROC: ",score['test_roc_auc'].mean()*100)

precision:  99.55241373384347
recall:  99.65289428654899
f1:  99.60202572312494
ROC:  99.46029615341911


**Training-Test Split for Red and White Dataframes**

In [ ]:
X_red = red_df.sample(frac=1)
X_red = red_df.drop(columns = ['type','quality'])
X_white = white_df.sample(frac=1)
X_white = white_df.drop(columns = ['type','quality'])
y_red_quality = red_df['quality']
y_white_quality = white_df['quality']

# red quality splitting
quality_X_red_train, quality_X_red_test, quality_y_red_train, quality_y_red_test = train_test_split(X_red,y_red_quality,test_size=0.2)

# white quality splitting
quality_X_white_train, quality_X_white_test, quality_y_white_train, quality_y_white_test = train_test_split(X_white,y_white_quality,test_size=0.2)

**Feature Scaling for Red and White Dataframe**

In [ ]:
scaler=StandardScaler()
scaler_X_red=scaler.fit_transform(X_red)
scaler_X_white=scaler.fit_transform(X_white)
scaler_train_red=scaler.fit_transform(quality_X_red_train)
scaler_test_red=scaler.fit_transform(quality_X_red_test)
scaler_train_white=scaler.fit_transform(quality_X_white_train)
scaler_test_white=scaler.fit_transform(quality_X_white_test)

**Random Forest - Quality Classification**

Red Wine

In [ ]:
RF_clf = RandomForestClassifier()

RF_clf.fit(quality_X_red_train, quality_y_red_train)

y_pred_red_1 = RF_clf.predict(quality_X_red_test)
y_pred_proba_red = RF_clf.predict_proba(quality_X_red_test)
y_pred_red = np.argmax(y_pred_proba_red, axis=1)

precision = precision_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba_red, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_red_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.6600028433135845
Recall: 0.678125
F1 score: 0.6658762246398934
ROC AUC score: 0.8480077732943752
accuracy:  0.678125


In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

RF_clf = RandomForestClassifier()

score = cross_validate(RF_clf,scaler_X_red,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  67.44250556683062
recall:  70.29559748427671
f1:  68.30290536950605
ROC:  85.00617690307021


White Wine

In [ ]:
RF_clf = RandomForestClassifier()

RF_clf.fit(quality_X_white_train, quality_y_white_train)

y_pred_white_1 = RF_clf.predict(quality_X_white_test)
y_pred_proba_white = RF_clf.predict_proba(quality_X_white_test)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)

precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.681908420396336
Recall: 0.6714285714285714
F1 score: 0.6613650605827895
ROC AUC score: 0.859402513296742
accuracy:  0.6714285714285714


In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

RF_clf = RandomForestClassifier()

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']

score = cross_validate(RF_clf,X_white_cv,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  71.10734139890637
recall:  70.36571929385251
f1:  69.30143249483687
ROC:  86.51783804677432


**Random Forest - PCA - Red Wine**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=11)
X_train_red_pca = pca.fit_transform(quality_X_red_train)
X_test_red_pca = pca.transform(quality_X_red_test)

rfc = RandomForestClassifier()
rfc.fit(X_train_red_pca, quality_y_red_train)

y_pred_1 = rfc.predict(X_test_red_pca)
y_pred_proba = rfc.predict_proba(X_test_red_pca)
y_pred = np.argmax(y_pred_proba, axis=1)

precision = precision_score(quality_y_red_test, y_pred_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.6562888244371681
Recall: 0.678125
F1 score: 0.6613849646911913
ROC AUC score: 0.8417703644096136
accuracy:  0.678125


In [ ]:
pca = PCA(n_components=11)

X_red_pca = pca.fit_transform(X_red)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

RF_clf = RandomForestClassifier(n_estimators=200,criterion='entropy',random_state=42)

score = cross_validate(RF_clf,X_red_pca,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  69.9589370452481
recall:  72.60888364779873
f1:  70.4105884307767
ROC:  85.99243810767734


**Random Forest - PCA - White Wine**

In [ ]:
pca = PCA(n_components=11)
X_train_white_pca = pca.fit_transform(quality_X_white_train)
X_test_white_pca = pca.transform(quality_X_white_test)

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train_white_pca, quality_y_white_train)

y_pred_white_1 = rfc.predict(X_test_white_pca)
y_pred_proba_white = rfc.predict_proba(X_test_white_pca)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)


precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)


Precision: 0.6869630842001598
Recall: 0.6775510204081633
F1 score: 0.6674682880133783
ROC AUC score: 0.8460061989602584
accuracy:  0.6775510204081633


In [ ]:
pca = PCA(n_components=11)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

RF_clf = RandomForestClassifier()

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']
X_white_cv_pca = pca.fit_transform(X_white_cv)

score = cross_validate(RF_clf,X_white_cv_pca,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  70.80055401597073
recall:  69.38562664329535
f1:  68.30212498768373
ROC:  84.32237247284871


**SVM - Quality Classification**

Red wine

In [ ]:
SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

SVC_clf.fit(quality_X_red_train, quality_y_red_train)

y_pred_red_1 = SVC_clf.predict(quality_X_red_test)
y_pred_proba_red = SVC_clf.predict_proba(quality_X_red_test)
y_pred_red = np.argmax(y_pred_proba_red, axis=1)

precision = precision_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba_red, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_red_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5661896654968805
Recall: 0.2625
F1 score: 0.3286091735433771
ROC AUC score: 0.7222243117002012
accuracy:  0.2625


In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

score = cross_validate(SVC_clf,X_red,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  56.825902070532884
recall:  29.32822327044025
f1:  35.97374048395877
ROC:  75.82152880258035


White wine

In [ ]:
SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

SVC_clf.fit(quality_X_white_train, quality_y_white_train)

y_pred_white_1 = SVC_clf.predict(quality_X_white_test)
y_pred_proba_white = SVC_clf.predict_proba(quality_X_white_test)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)

precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.4475502519390819
Recall: 0.2642857142857143
F1 score: 0.2904950980323586
ROC AUC score: 0.6739899276363671
accuracy:  0.2642857142857143


In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']

score = cross_validate(SVC_clf,X_white_cv,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  44.758721812936635
recall:  28.96001836317349
f1:  29.502671613376652
ROC:  70.73853869479566


**SVM - PCA - Red Wine**

In [ ]:
pca = PCA(n_components=11)
X_train_red_pca = pca.fit_transform(quality_X_red_train)
X_test_red_pca = pca.transform(quality_X_red_test)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)
SVC_clf.fit(X_train_red_pca, quality_y_red_train)

y_pred_red_1 = SVC_clf.predict(X_test_red_pca)
y_pred_proba_red = SVC_clf.predict_proba(X_test_red_pca)
y_pred_red = np.argmax(y_pred_proba_red, axis=1)

precision = precision_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba_red, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_red_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5209778316408811
Recall: 0.24375
F1 score: 0.31796951235403836
ROC AUC score: 0.6858065128705872
accuracy:  0.24375


In [ ]:
pca = PCA(n_components=11)

X_red_pca = pca.fit_transform(X_red)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

score = cross_validate(SVC_clf,X_red_pca,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  56.825902070532884
recall:  29.32822327044025
f1:  35.97374048395877
ROC:  75.41152584074888


**SVM - PCA - White Wine**

In [ ]:
pca = PCA(n_components=11)
X_train_white_pca = pca.fit_transform(quality_X_white_train)
X_test_white_pca = pca.transform(quality_X_white_test)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)
SVC_clf.fit(X_train_white_pca, quality_y_white_train)

y_pred_white_1 = SVC_clf.predict(X_test_white_pca)
y_pred_proba_white = SVC_clf.predict_proba(X_test_white_pca)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)


precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.4642294290889156
Recall: 0.25612244897959185
F1 score: 0.273551713044569
ROC AUC score: 0.6816593236312414
accuracy:  0.25612244897959185


In [ ]:
pca = PCA(n_components=11)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

SVC_clf = SVC(C=0.01, kernel='linear', class_weight='balanced', probability=True)

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']
X_white_cv_pca = pca.fit_transform(X_white_cv)

score = cross_validate(SVC_clf,X_white_cv_pca,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  44.767134099103565
recall:  28.980468260924003
f1:  29.51767517821704
ROC:  70.25076639471658


**K-Nearest Neighbors - Quality Classification**

Red Wine

In [ ]:
KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')

KNN_clf.fit(quality_X_red_train, quality_y_red_train)

y_pred_red_1 = KNN_clf.predict(quality_X_red_test)
y_pred_proba_red = KNN_clf.predict_proba(quality_X_red_test)
y_pred_red = np.argmax(y_pred_proba_red, axis=1)

precision = precision_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba_red, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_red_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5702985059155241
Recall: 0.578125
F1 score: 0.5739686878130367
ROC AUC score: 0.6702022623699784
accuracy:  0.578125


In [ ]:
scaler_X_KNN_red=scaler.fit_transform(X_red)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')

score = cross_validate(KNN_clf,scaler_X_KNN_red,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  64.37347536375185
recall:  64.5440251572327
f1:  64.24993002921347
ROC:  63.56277750418059


White Wine

In [ ]:
KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')
KNN_clf.fit(quality_X_white_train, quality_y_white_train)

y_pred_white_1 = KNN_clf.predict(quality_X_white_test)
y_pred_proba_white = KNN_clf.predict_proba(quality_X_white_test)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)

precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5718686459511217
Recall: 0.5704081632653061
F1 score: 0.5702475963043269
ROC AUC score: 0.6823334156601115
accuracy:  0.5704081632653061


In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']
scaler_X_KNN_white_cv=scaler.fit_transform(X_white_cv)

score = cross_validate(KNN_clf,scaler_X_KNN_white_cv,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  65.64336486496684
recall:  65.8901548349401
f1:  65.66901814830467
ROC:  69.89360817136337


**KNN - PCA - Red Wine**

In [ ]:
pca = PCA(n_components=11)
X_train_red_pca = pca.fit_transform(quality_X_red_train)
X_test_red_pca = pca.transform(quality_X_red_test)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')
KNN_clf.fit(X_train_red_pca, quality_y_red_train)

y_pred_red_1 = KNN_clf.predict(X_test_red_pca)
y_pred_proba_red = KNN_clf.predict_proba(X_test_red_pca)
y_pred_red = np.argmax(y_pred_proba_red, axis=1)

precision = precision_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_red_test, y_pred_red_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_red_test, y_pred_proba_red, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_red_test, y_pred_red_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5901411652135246
Recall: 0.615625
F1 score: 0.5997076162157997
ROC AUC score: 0.7800392738286913
accuracy:  0.615625


In [ ]:
pca = PCA(n_components=11)

X_red_pca = pca.fit_transform(scaler_X_red)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')

score = cross_validate(KNN_clf,X_red_pca,y_red_quality,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  64.37347536375185
recall:  64.5440251572327
f1:  64.24993002921347
ROC:  63.56277750418059


**KNN - PCA - White Wine**

In [ ]:
pca = PCA(n_components=11)
X_train_white_pca = pca.fit_transform(quality_X_white_train)
X_test_white_pca = pca.transform(quality_X_white_test)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')
KNN_clf.fit(X_train_white_pca, quality_y_white_train)

y_pred_white_1 = KNN_clf.predict(X_test_white_pca)
y_pred_proba_white = KNN_clf.predict_proba(X_test_white_pca)
y_pred_white = np.argmax(y_pred_proba_white, axis=1)


precision = precision_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
recall = recall_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
f1 = f1_score(quality_y_white_test, y_pred_white_1, average='weighted', zero_division=0)
roc_auc = roc_auc_score(quality_y_white_test, y_pred_proba_white, average='weighted', multi_class='ovr')
accuracy = accuracy_score(quality_y_white_test, y_pred_white_1)

print('Precision:', precision)
print('Recall:', recall)
print('F1 score:', f1)
print('ROC AUC score:', roc_auc)
print("accuracy: ", accuracy)

Precision: 0.5982425588730759
Recall: 0.5989795918367347
F1 score: 0.5980470642227519
ROC AUC score: 0.6961730497587342
accuracy:  0.5989795918367347


In [ ]:
pca = PCA(n_components=11)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

KNN_clf = KNeighborsClassifier(n_neighbors=1,weights='distance')

special_df = white_df[white_df.quality != 9]
X_white_cv = special_df.drop(columns = ['type','quality'])
y_white_quality_cv = special_df['quality']
X_white_cv = scaler.fit_transform(X_white_cv)
X_white_cv_pca = pca.fit_transform(X_white_cv)

score = cross_validate(KNN_clf,X_white_cv_pca,y_white_quality_cv,scoring=['precision_weighted','recall_weighted','f1_weighted','roc_auc_ovr'],cv=kfold)
print("precision: ",score['test_precision_weighted'].mean()*100)
print("recall: ",score['test_recall_weighted'].mean()*100)
print("f1: ",score['test_f1_weighted'].mean()*100)
print("ROC: ",score['test_roc_auc_ovr'].mean()*100)

precision:  65.64336486496684
recall:  65.8901548349401
f1:  65.66901814830467
ROC:  69.89360817136337
